[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-regularization.ipynb)

# Regularization — Ridge, Lasso & ElasticNet

*AIBits Academy · Machine Learning End To End · Supervised Learning · New*

Penalising model complexity to fight overfitting — the single most important technique for making linear models generalise reliably.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **🎯 Intuition First**
>
> Left unchecked, a linear model happily cranks its weights to extremes to chase every last training point — then falls apart on new data. Regularization is simply a **leash**: a penalty that makes large weights "expensive," forcing the model to keep them small unless a feature genuinely earns its keep.

> **📋 Real-World Case Study — Municipal Bond Investment Risk**
>
> A municipal bond dataset (1,198 records) had 55 raw columns — after one-hot encoding a handful of high-cardinality fields like Underwriter and Bond Counsel, that became **1,302 columns on 1,198 rows**. Plain Linear Regression didn't just underperform — its R² collapsed to **−2,545,768**, a numerical breakdown from a near-singular feature matrix. Adding a single Ridge penalty, with zero other changes, restored R² to **0.598**. Lasso reached 0.480. Nothing about the data or features changed — only the addition of a complexity penalty stood between a catastrophically broken model and a usable one.

## Getting a Feel for It First — Bias, Variance & the Sweet Spot

Before the formula, the intuition. Imagine you're an archer, and each arrow is a model trained on a slightly different sample of data. Two things can go wrong:

- **Bias** is being *consistently off-target* — every arrow lands in a tight cluster, but far from the bullseye. The model is too simple to capture the real pattern (**underfitting**): it makes the same wrong assumption every time, no matter what data it sees.
- **Variance** is being *wildly inconsistent* — the arrows scatter all over the board. The model is so complex it chases every random wobble in the training data (**overfitting**): change the sample slightly and it lands somewhere completely different.

A model that's too simple has high bias; a model that's too complex has high variance. Crucially, you usually *can't* drive both to zero at once — pushing complexity down to kill variance raises bias, and pushing it up to kill bias raises variance. That tension is the **bias–variance trade-off**, and the whole game is finding the complexity that minimises their *sum*.

The top panel says the same thing with curves instead of dartboards. As complexity grows left to right, the bias (train error, orange) keeps falling — a more flexible model always fits the training data better. But variance (blue) climbs, because that same flexibility starts memorising noise. Their sum, the generalization (test) error, red, is therefore **U-shaped**: it drops, bottoms out at the optimal complexity band, then rises again as overfitting takes over. Everything left of that band is underfitting; everything right of it is overfitting.

> **💡 Where Regularization Fits In**
>
> Regularization is a *dial* for exactly this trade-off. Instead of changing the model type, it adds a penalty that gently pulls a too-flexible model back toward simplicity — sliding you leftward along the curve, out of the overfitting zone and toward that optimal band. The rest of this page makes that precise. First, the formal decomposition that names the three pieces of error:

## Why Regularize? The Bias-Variance Trade-Off, Formally

Every model's expected test error decomposes into three additive parts. For a fitted model f̂ predicting true function f at a point x:

$$E\big[(y-\hat{f}(x))^2\big] = \text{Bias}[\hat{f}(x)]^2 + \text{Var}[\hat{f}(x)] + \sigma^2_{\text{irreducible}}$$

An unregularized linear model fit on many correlated or noisy features has **low bias but high variance** — small changes in the training set (a few different Ahmedabad apartments in the sample) swing the coefficients wildly. Regularization deliberately introduces a small amount of *bias* in exchange for a large reduction in *variance*, usually lowering total test error.

## Ridge Regression (L2 Penalty)

Ridge adds the sum of squared coefficients to the cost function:

$$J(\theta) = \frac{1}{2m}\lVert\mathbf{X}\theta-\mathbf{y}\rVert^2 \;+\; \lambda\sum_{j=1}^{p}\theta_j^2$$

Unlike ordinary least squares, Ridge has a closed-form solution even when X is rank-deficient or multicollinear, because adding λI makes the matrix invertible:

$$\theta_{\text{ridge}} = (\mathbf{X}^{\top}\mathbf{X}+\lambda\mathbf{I})^{-1}\mathbf{X}^{\top}\mathbf{y}$$

Note the bias term θ₀ is conventionally *excluded* from the penalty (only θ₁…θₚ are shrunk) — shrinking the intercept would bias predictions toward zero regardless of scale.

## Lasso Regression (L1 Penalty)

$$J(\theta) = \frac{1}{2m}\lVert\mathbf{X}\theta-\mathbf{y}\rVert^2 \;+\; \lambda\sum_{j=1}^{p}|\theta_j|$$

Lasso has no closed-form solution (the |·| term is not differentiable at 0) — it's solved via coordinate descent or subgradient methods. The critical practical difference from Ridge: Lasso drives some coefficients to **exactly zero**, performing automatic feature selection.

## Why L1 Zeroes Coefficients but L2 Doesn't — The Geometry

Both penalised problems can be rewritten as constrained optimisation: minimise the unpenalised MSE subject to a budget on the coefficients (‖θ‖₁ ≤ t for Lasso, ‖θ‖₂² ≤ t for Ridge). The MSE's elliptical contours expand outward from the OLS solution until they touch the constraint region's boundary.

The diamond's **corners** lie on the coordinate axes (where one coefficient is exactly 0). Elliptical MSE contours are far more likely to first touch a corner than a smooth curve is — hence Lasso sparsity. The circle has no corners, so Ridge shrinks every coefficient toward (but essentially never exactly to) zero.

## ElasticNet — Best of Both

$$J(\theta) = \frac{1}{2m}\lVert\mathbf{X}\theta-\mathbf{y}\rVert^2 \;+\; \lambda\Big[r\sum|\theta_j| + \frac{1-r}{2}\sum\theta_j^2\Big]$$

The mixing ratio **r ∈ [0,1]** (sklearn's `l1_ratio`) interpolates between pure Lasso (r=1) and pure Ridge (r=0). ElasticNet is preferred when features are correlated in *groups* — Lasso alone tends to arbitrarily pick one feature from a correlated group and zero out the rest, while ElasticNet's L2 component keeps correlated groups together, shrinking them jointly rather than arbitrarily eliminating members.

## Worked Example — Ahmedabad Apartments, Extended with Noise Features

We reuse the Multiple Regression dataset but add 6 irrelevant/noisy columns to see how each penalty responds:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 500
area   = np.random.randint(500, 2500, n)
floors = np.random.randint(1, 25, n)
age    = np.random.randint(0, 30, n)
dist   = np.round(np.random.uniform(0.2, 8, n), 1)
# 6 irrelevant noise columns — none affect price
noise  = np.random.normal(0, 1, (n, 6))
price  = (0.08*area + 1.5*floors - 1.2*age - 3.5*dist
          + np.random.normal(0, 8, n) + 40)

cols = ['area','floors','age','dist_brts'] + [f'noise_{i}' for i in range(6)]
X = pd.DataFrame(np.column_stack([area,floors,age,dist,noise]), columns=cols)
y = price

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler(); X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)

models = {
    'OLS':       LinearRegression(),
    'Ridge α=5': Ridge(alpha=5),
    'Lasso α=0.5': Lasso(alpha=0.5),
    'ElasticNet r=0.5': ElasticNet(alpha=0.5, l1_ratio=0.5),
}
for name, m in models.items():
    m.fit(X_tr_s, y_tr)
    zeros = np.sum(np.abs(m.coef_) < 1e-3)
    print(f"{name:18s}  Test R²={m.score(X_te_s,y_te):.4f}  #coefs≈0={zeros}/10")

# Coefficient magnitudes on the 6 noise features specifically
lasso = Lasso(alpha=0.5).fit(X_tr_s, y_tr)
print("\nLasso noise-feature coefficients:", np.round(lasso.coef_[4:], 4))

Lasso correctly zeroes out **all 6 noise features** while OLS assigns them small nonzero (spurious) coefficients that would mislead interpretation. This is the essence of Lasso as a feature-selection tool.

## Choosing λ — Regularization Path & Cross-Validation

λ (sklearn's `alpha`) is a hyperparameter — never fit it on the training set directly. Use `RidgeCV` / `LassoCV`, which internally cross-validate a grid of λ values:

In [ ]:
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5)
ridge_cv.fit(X_tr_s, y_tr)
print(f"Best Ridge alpha: {ridge_cv.alpha_:.4f}")

lasso_cv = LassoCV(alphas=np.logspace(-3, 1, 50), cv=5, random_state=42)
lasso_cv.fit(X_tr_s, y_tr)
print(f"Best Lasso alpha: {lasso_cv.alpha_:.4f}")
# As alpha increases from 0 → ∞: coefficients shrink continuously toward 0
# (Ridge) or hit 0 one-by-one in order of decreasing importance (Lasso)

## Effect of λ — Quick Reference

| λ value | Effect | Risk |
|---|---|---|
| λ → 0 | Reduces to OLS | High variance, overfitting on noisy/correlated features |
| λ small–moderate | Mild shrinkage, sweet spot found via CV | Balanced bias-variance |
| λ → ∞ | All coefficients → 0 (Ridge) or exactly 0 (Lasso) | High bias, underfitting — model predicts the mean |

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Ridge shrinks coefficients

Fit `LinearRegression` and `Ridge(alpha=10)` on the data below. Store the coefficient vectors as `ols_c` and `ridge_c`, and `shrinks` = whether Ridge's coefficient vector has the smaller norm.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
rng = np.random.default_rng(1)
X = rng.normal(size=(40, 6))
y = X @ np.array([3, -2, 0, 0, 1, 0]) + rng.normal(scale=2, size=40)
ols_c = ridge_c = shrinks = None   # TODO


In [ ]:
try:
    check("Ridge has smaller norm", shrinks is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
rng = np.random.default_rng(1)
X = rng.normal(size=(40, 6))
y = X @ np.array([3, -2, 0, 0, 1, 0]) + rng.normal(scale=2, size=40)
ols_c = LinearRegression().fit(X, y).coef_
ridge_c = Ridge(alpha=10).fit(X, y).coef_
shrinks = bool(np.linalg.norm(ridge_c) < np.linalg.norm(ols_c))

```

</details>

### Exercise 2 · Medium · Lasso performs feature selection

Only 3 of the 10 features matter. Fit `Lasso(alpha=0.5)` and store the **number of coefficients that are exactly zero** in `n_zero`.

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso
rng = np.random.default_rng(2)
X = rng.normal(size=(200, 10))
y = 4 * X[:, 0] - 3 * X[:, 1] + 2 * X[:, 2] + rng.normal(size=200)
n_zero = None   # TODO


In [ ]:
try:
    check("most noise features are zeroed", n_zero >= 6)
    check("but not everything", n_zero <= 7)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.linear_model import Lasso
rng = np.random.default_rng(2)
X = rng.normal(size=(200, 10))
y = 4 * X[:, 0] - 3 * X[:, 1] + 2 * X[:, 2] + rng.normal(size=200)
n_zero = int((Lasso(alpha=0.5).fit(X, y).coef_ == 0).sum())

```

</details>

### Exercise 3 · Stretch · Pick alpha by cross-validation

Use `RidgeCV(alphas=np.logspace(-3, 3, 30), cv=5)` **inside a pipeline with `StandardScaler`**. Fit on the training split, and store the chosen alpha in `best_alpha` and the test R² in `test_r2`.

In [ ]:
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(5)
X = rng.normal(size=(300, 8)) * rng.uniform(1, 50, 8)
y = X[:, 0] * 0.1 - X[:, 3] * 0.05 + rng.normal(size=300)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
best_alpha = test_r2 = None   # TODO


In [ ]:
try:
    check("alpha chosen from the grid", 1e-3 <= best_alpha <= 1e3)
    check("test R2 is decent", test_r2 > 0.5)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(5)
X = rng.normal(size=(300, 8)) * rng.uniform(1, 50, 8)
y = X[:, 0] * 0.1 - X[:, 3] * 0.05 + rng.normal(size=300)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
pipe = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 30), cv=5)).fit(X_tr, y_tr)
best_alpha = pipe[-1].alpha_
test_r2 = pipe.score(X_te, y_te)

```

Scaling first matters: the Ridge penalty treats all coefficients alike, so features must be on comparable scales.

</details>

---
*Back to the course: **Machine Learning End To End → Regularization — Ridge, Lasso & ElasticNet**.*